# 02 — Emotion Classifier

Train two text classifiers (Logistic Regression and LinearSVC) on TF-IDF features,
compare them, and save the winner for use in the Spotify pipeline.

In [ ]:
import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "figure.facecolor": "white",
                     "axes.spines.top": False, "axes.spines.right": False})
%matplotlib inline

DATA_DIR   = Path("data/processed")
MODELS_DIR = Path("models")
OUTPUT_DIR = Path("outputs")
MODELS_DIR.mkdir(exist_ok=True)

In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

# load the macro map so label ordering is consistent
with open(DATA_DIR / "macro_map.json") as f:
    macro_map = json.load(f)

CLASSES = sorted(train_df["label"].unique())
print(f"Classes ({len(CLASSES)}): {CLASSES}")
print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")

In [ ]:
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)          # strip URLs
    text = re.sub(r"[^a-z\s']", " ", text)       # keep letters, spaces, apostrophes
    text = re.sub(r"\s+", " ", text).strip()
    return text

X_train = train_df["text"].map(clean_text)
X_val   = val_df["text"].map(clean_text)
X_test  = test_df["text"].map(clean_text)

y_train = train_df["label"]
y_val   = val_df["label"]
y_test  = test_df["label"]

print("Cleaning done. Sample:")
print(f"  raw : {train_df['text'].iloc[0]}")
print(f"  clean: {X_train.iloc[0]}")

In [ ]:
# unigrams + bigrams; sublinear_tf dampens the effect of very frequent terms
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=25_000,
    sublinear_tf=True,
    min_df=3,
    strip_accents="unicode",
    stop_words="english",
)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec   = vectorizer.transform(X_val)
X_test_vec  = vectorizer.transform(X_test)

print(f"Vocabulary size: {len(vectorizer.vocabulary_):,}")
print(f"Train matrix: {X_train_vec.shape}")

In [ ]:
# --- Logistic Regression ---
lr = LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced", random_state=42, n_jobs=-1)
lr.fit(X_train_vec, y_train)

lr_val_preds = lr.predict(X_val_vec)
lr_val_f1    = f1_score(y_val, lr_val_preds, average="weighted")
print(f"LogReg — val weighted F1: {lr_val_f1:.4f}")

In [ ]:
# --- LinearSVC (wrapped in CalibratedClassifierCV so we get predict_proba later) ---
base_svc = LinearSVC(C=0.3, max_iter=3000, class_weight="balanced")
svc = CalibratedClassifierCV(base_svc, cv=3)
svc.fit(X_train_vec, y_train)

svc_val_preds = svc.predict(X_val_vec)
svc_val_f1    = f1_score(y_val, svc_val_preds, average="weighted")
print(f"LinearSVC — val weighted F1: {svc_val_f1:.4f}")

In [ ]:
# pick the better model by val F1 and evaluate it on the held-out test set
if svc_val_f1 >= lr_val_f1:
    best_model, best_name, test_preds = svc, "LinearSVC", svc.predict(X_test_vec)
else:
    best_model, best_name, test_preds = lr,  "LogReg",    lr.predict(X_test_vec)

print(f"Best model: {best_name}\n")
print(classification_report(y_test, test_preds, target_names=CLASSES, digits=3))

In [ ]:
cm = confusion_matrix(y_test, test_preds, labels=CLASSES)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=CLASSES, yticklabels=CLASSES,
            cmap="Blues", linewidths=0.5, ax=ax)
ax.set_title(f"Confusion matrix — {best_name} (test set)", fontsize=13)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.tick_params(axis="x", rotation=40)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", bbox_inches="tight")
plt.show()

In [ ]:
# per-class F1 bar chart — easier to read than the confusion matrix for imbalanced data
report = pd.DataFrame(
    classification_report(y_test, test_preds, target_names=CLASSES, output_dict=True)
).T.loc[CLASSES, ["precision", "recall", "f1-score"]]

fig, ax = plt.subplots(figsize=(11, 4))
report["f1-score"].sort_values().plot(kind="barh", ax=ax, color="steelblue", edgecolor="none")
ax.axvline(0.5, color="red", linestyle="--", linewidth=1, label="0.50 threshold")
ax.set_title(f"Per-class F1 — {best_name}", fontsize=13)
ax.set_xlabel("F1 score")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "per_class_f1.png", bbox_inches="tight")
plt.show()

In [ ]:
# save the vectorizer and best model together so the pipeline loads one file each
joblib.dump(vectorizer, MODELS_DIR / "tfidf_vectorizer.pkl", compress=3)
joblib.dump(best_model, MODELS_DIR / "emotion_classifier.pkl", compress=3)

meta = {"model": best_name, "val_f1": round(max(lr_val_f1, svc_val_f1), 4), "classes": CLASSES}
with open(MODELS_DIR / "model_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"Saved vectorizer + {best_name} to models/")
print(json.dumps(meta, indent=2))